# Task 1

In [1]:
import nltk
from typing import List, Tuple

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/russele7/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/russele7/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    """
    Токенизирует текст и возвращает список кортежей (token, start_char, end_char).
    """
    tokens = nltk.word_tokenize(text)
    
    offsets = []
    pos = 0  # с какой позиции искать следующий токен
    
    for tok in tokens:
        # Найти начальную позицию токена в тексте, начиная с позиции pos
        # Используйте метод text.find(tok, pos)
        start = text.find(tok, pos) # Ваш код здесь
        
        # Вычислить конечную позицию (начало + длина токена)
        end = start + len(tok) # Ваш код здесь
        
        # Добавить кортеж (токен, начало, конец) в список результатов
        offsets.append((tok, start, end)) # Ваш код здесь)
        
        # Обновить позицию для поиска следующего токена
        # Следующий токен будем искать после текущего
        pos = end # Ваш код здесь
    
    return offsets

In [4]:
# Тест
test_text = "Иван Петров работает."
result = tokenize_with_offsets(test_text)

In [5]:
print("Результат:")
for token, start, end in result:
    print(f"'{token}' -> [{start}-{end})") 

Результат:
'Иван' -> [0-4)
'Петров' -> [5-11)
'работает' -> [12-20)
'.' -> [20-21)


## Task 2

In [6]:
from typing import List, Tuple

In [7]:

def spans_to_bio(tokens_off: List[Tuple[str, int, int]], 
                 spans: List[Tuple[int, int, str]]) -> List[str]:
    """
    Преобразует спаны именованных сущностей в BIO-метки для токенов.
    """
    # Инициализируем все метки как 'O' (Outside)
    bio = ['O'] * len(tokens_off) # Ваш код здесь - создайте список из 'O' длиной, равной количеству токенов
    
    # Обрабатываем каждый спан сущности
    for span_start, span_end, label in spans:
        first_token_in_span = True  # флаг для отслеживания первого токена в спане
        
        # Проверяем каждый токен на пересечение с текущим спаном
        for i, (token, t_start, t_end) in enumerate(tokens_off):
            
            # Проверяем, пересекается ли токен со спаном
            # Условие НЕпересечения: токен полностью до спана ИЛИ полностью после спана
            if t_end <= span_start or t_start >= span_end: # Ваш код здесь - условие: t_end <= span_start ИЛИ t_start >= span_end:
                continue  # токен не пересекается со спаном, переходим к следующему
                
            # Токен пересекается со спаном!
            if first_token_in_span:
                # Это первый токен в данном спане - помечаем как B-LABEL
                bio[i] = f"B-{label}" # Ваш код здесь - создайте строку вида "B-{label}"
                first_token_in_span = False
            else:
                # Это НЕ первый токен в спане - помечаем как I-LABEL
                bio[i] = f"I-{label}" # Ваш код здесь - создайте строку вида "I-{label}"
    
    return bio

In [8]:
# Простой тест
# Сначала нужна функция tokenize_with_offsets из предыдущего задания
def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    import nltk
    tokens = nltk.word_tokenize(text)
    offsets = []
    pos = 0
    for tok in tokens:
        start = text.find(tok, pos)
        end = start + len(tok)
        offsets.append((tok, start, end))
        pos = end
    return offsets

In [9]:
# Тест
text = "Иван Иванов работает в Яндексе."
spans = [(0, 11, "PER"), (23, 29, "ORG")]  # "Иван Иванов" и "Яндекс"

tokens_off = tokenize_with_offsets(text)
bio_tags = spans_to_bio(tokens_off, spans)

print(f'tokens_off = {tokens_off}')
print(f'bio_tags = {bio_tags}')

print("Результат BIO-разметки:")
for (token, start, end), bio_tag in zip(tokens_off, bio_tags):
    print(f"{token:12} -> {bio_tag}") 

tokens_off = [('Иван', 0, 4), ('Иванов', 5, 11), ('работает', 12, 20), ('в', 21, 22), ('Яндексе', 23, 30), ('.', 30, 31)]
bio_tags = ['B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'O']
Результат BIO-разметки:
Иван         -> B-PER
Иванов       -> I-PER
работает     -> O
в            -> O
Яндексе      -> B-ORG
.            -> O


# Task 3

In [13]:
from typing import List

In [14]:
def bio_to_bioes(bio: List[str]) -> List[str]:
    """
    Преобразует BIO-метки в BIOES-метки.
    Параметры:
    - bio: список BIO-меток (строки вида 'B-LABEL', 'I-LABEL', 'O')
    Возвращает:
    - Список BIOES-меток (строки вида 'B-LABEL', 'I-LABEL', 'E-LABEL', 'S-LABEL', 'O')
    """
    
    bioes = []
    n = len(bio)
    
    for i, tag in enumerate(bio):
        if tag == 'O':
            # O-теги остаются без изменений
            bioes.append('O')
            continue
            
        # Разбираем тег на префикс (B/I) и лейбл (PER/ORG/LOC/...)
        # Используйте метод split с параметром maxsplit=1 для корректной обработки 
        # лейблов, содержащих дефис (например, 'B-PERSON-NAME')
        prefix, label = tag.split('-', 1)
        
        if prefix == 'B':
            # B-тег: проверяем, что идёт после него
            
            # Проверяем условия:
            # 1. Есть ли следующий элемент в списке (i+1 < n)
            # 2. Является ли следующий элемент продолжением той же сущности (I-{label})
            if i + 1 < n and bio[i + 1] == f'I-{label}':
                # После B идёт I того же лейбла → это начало многотокенной сущности
                bioes.append(f'B-{label}')
            else:
                # После B НЕ идёт I того же лейбла → это однотокенная сущность
                bioes.append(f'S-{label}')
                
        elif prefix == 'I':
            # I-тег: проверяем, что идёт после него
            
            # Аналогично проверяем следующий элемент
            if i + 1 < n and bio[i + 1] == f'I-{label}':
                # После I идёт ещё I того же лейбла → это середина многотокенной сущности  
                bioes.append(f'I-{label}')
            else:
                # После I НЕ идёт I того же лейбла → это конец многотокенной сущности
                bioes.append(f'E-{label}')
        else:
            # Неожиданный префикс (не B и не I) - оставляем как есть
            bioes.append(tag)
    
    return bioes

In [15]:


# Тест всех функций вместе
def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    import nltk
    tokens = nltk.word_tokenize(text)
    offsets = []
    pos = 0
    for tok in tokens:
        start = text.find(tok, pos)
        end = start + len(tok)
        offsets.append((tok, start, end))
        pos = end
    return offsets

def spans_to_bio(tokens_off, spans):
    bio = ['O'] * len(tokens_off)
    for span_start, span_end, label in spans:
        first_token_in_span = True
        for i, (token, t_start, t_end) in enumerate(tokens_off):
            if t_end <= span_start or t_start >= span_end:
                continue
            if first_token_in_span:
                bio[i] = f'B-{label}'
                first_token_in_span = False
            else:
                bio[i] = f'I-{label}'
    return bio

# Полный тест
text = "Иван Иванович Петров работает в Google."
spans = [(0, 20, "PER"), (31, 37, "ORG")]  # длинная персона и короткая организация

tokens_off = tokenize_with_offsets(text)
bio_tags = spans_to_bio(tokens_off, spans)
bioes_tags = bio_to_bioes(bio_tags)

print("Сравнение BIO и BIOES:")
for (token, _, _), bio_tag, bioes_tag in zip(tokens_off, bio_tags, bioes_tags):
    print(f"{token:12} {bio_tag:8} -> {bioes_tag:8}")

Сравнение BIO и BIOES:
Иван         B-PER    -> B-PER   
Иванович     I-PER    -> I-PER   
Петров       I-PER    -> E-PER   
работает     O        -> O       
в            O        -> O       
Google       B-ORG    -> S-ORG   
.            O        -> O       


In [16]:
def compute_f1(predicted, actual):
    # predicted и actual - списки спанов вида (start, end, label)
    pred_set = set(predicted)
    actual_set = set(actual)
    true_positives = len(pred_set & actual_set)
    if true_positives == 0:
        return 0.0
    precision = true_positives / len(pred_set)
    recall = true_positives / len(actual_set)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

# Пример
pred = [(0,4,"PER"), (17,21,"LOC")]    # Джек (верно), "Нью" (LOC, неверно)
actual = [(0,4,"PER"), (17,26,"LOC")] # Джек, "Нью-Йорк"
print(compute_f1(pred, actual))  # 0.5 

0.5


# Task 4

In [18]:
raw_ner = [
    {'word': 'Газ', 'entity': 'B-ORG', 'score': 0.95, 'start': 10, 'end': 13},
    {'word': '##проме', 'entity': 'I-ORG', 'score': 0.93, 'start': 13, 'end': 18},
    {'word': '-', 'entity': 'O', 'score': 0.00, 'start': 18, 'end': 19},
    {'word': 'он', 'entity': 'O', 'score': 0.00, 'start': 20, 'end': 22}
]

In [19]:
merged = []
for res in raw_ner:
    word = res['word']
    # Если слово начинается с '##', это subword-токен
    if word.startswith('##'): # Ваш код здесь
        # Склеиваем с предыдущим токеном
        # Ваш код здесь
        merged[-1]['word'] += word[2:]
        merged[-1]['end'] = res['end']

    else: # Если это не subword-токен, добавляем новый токен
        merged.append({'word': word, 'entity': res['entity'], 'start': res['start'], 'end': res['end']})

print(merged)
# Ожидаемый выход: [{'word': 'Газпроме', 'entity': 'B-ORG', 'start': 10, 'end': 18}, {'word': '-', 'entity': 'O', ...}, ...]

[{'word': 'Газпроме', 'entity': 'B-ORG', 'start': 10, 'end': 18}, {'word': '-', 'entity': 'O', 'start': 18, 'end': 19}, {'word': 'он', 'entity': 'O', 'start': 20, 'end': 22}]


# Task 5

In [20]:
from transformers import pipeline

/home/russele7/practicum/dle/sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
def merge_entities(ner_results):
    entities = []
    current = None
    for res in ner_results:
        word = res['word']
        label = res['entity']  # B-PER, I-PER, O и т.д.
        # Удаляем префикс 'B-', 'I-', 'S-', 'E-' чтобы получить тип
        ent_type = label.split('-')[-1] if label != 'O' else None

        if word.startswith('##'):  # subword-токен
            word = word[2:]
            # Добавляем к последнему слову
            if entities:
                entities[-1]['word'] += word
                entities[-1]['end'] = res['end']
            continue

        if label.startswith('B-') or label.startswith('S-'):
            # Начинается новая сущность (или единичная), через append добавляем новую сущность
            # Ваш код здесь
            entities.append({'word': word, 'type': ent_type, 'start': res['start'], 'end': res['end']})

        elif label.startswith('I-') or label.startswith('E-'):
            # Продолжаем предыдущую сущность
            if entities and entities[-1]['type'] == ent_type:
                # Добавляем к последнему слову с пробелом
                # Ваш код здесь
            
                entities[-1]['word'] += ' ' + word
                # Если сущность уже есть, обновляем её правую границу
                # Ваш код здесь
                entities[-1]['end'] = res['end']
            else:
                # Неожиданный случай - просто создаём новый
                entities.append({'word': word, 'type': ent_type, 'start': res['start'], 'end': res['end']})
        # Если 'O' - ничего не делаем
    return entities

In [ ]:
ner_pipeline = pipeline("ner", model="nesemenpolkov/msu-wiki-ner", tokenizer="nesemenpolkov/msu-wiki-ner")

In [22]:

sentence = "Иван Иванович Иванов работает в Газпроме."
raw_results = ner_pipeline(sentence)
print(raw_results)
merged = merge_entities(raw_results) # Ваш код здесь - вызов функции merge_entities
print("Склеенные сущности:", merged)

Device set to use cuda:0


[{'entity': 'B-PER', 'score': np.float32(0.99934477), 'index': 1, 'word': 'Иван', 'start': 0, 'end': 4}, {'entity': 'I-PER', 'score': np.float32(0.9998091), 'index': 2, 'word': 'Иванович', 'start': 5, 'end': 13}, {'entity': 'I-PER', 'score': np.float32(0.9998066), 'index': 3, 'word': 'Иванов', 'start': 14, 'end': 20}, {'entity': 'B-ORG', 'score': np.float32(0.9987931), 'index': 6, 'word': 'Г', 'start': 32, 'end': 33}, {'entity': 'I-ORG', 'score': np.float32(0.9985285), 'index': 7, 'word': '##аз', 'start': 33, 'end': 35}, {'entity': 'I-ORG', 'score': np.float32(0.9987846), 'index': 8, 'word': '##про', 'start': 35, 'end': 38}, {'entity': 'I-ORG', 'score': np.float32(0.99886453), 'index': 9, 'word': '##ме', 'start': 38, 'end': 40}]
Склеенные сущности: [{'word': 'Иван Иванович Иванов', 'type': 'PER', 'start': 0, 'end': 20}, {'word': 'Газпроме', 'type': 'ORG', 'start': 32, 'end': 40}]


In [ ]:
# ner_pipeline = pipeline(
#     "ner",
#     model="nesemenpolkov/msu-wiki-ner",
#     tokenizer="nesemenpolkov/msu-wiki-ner",
#     aggregation_strategy="simple"  # объединяет токены в спаны
# ) 